> **Public portfolio copy:** Saved execution outputs have been removed. The notebook source code and explanations are retained. Running the notebook requires a separately authorised DICOM input file in the local `Data/` folder. No Stanford COCA image or dataset file is included in this repository.

# DICOM Metadata Inspection and CT Image Analysis

## Project 2: DICOM Extension – Medical Image Viewer and Preprocessing

### Overview

This notebook explores the structure and technical characteristics of a CT image stored in DICOM format.

It demonstrates how to load a DICOM file, inspect its metadata, extract pixel data, convert raw pixel values into Hounsfield Units (HU), and examine image intensity characteristics.

### Objectives

- Load and inspect a DICOM dataset using Python.
- Examine important DICOM metadata and acquisition parameters.
- Extract and analyse the CT pixel array.
- Understand the conversion of raw pixel values into Hounsfield Units.
- Explore CT image intensity distributions and window settings.
- Perform regional pixel intensity analysis.
- Examine basic image data quality characteristics.

### Tools and Libraries

- Python
- Pydicom
- NumPy
- Matplotlib
- Jupyter Notebook

### Dataset

The notebook uses a single CT DICOM image (`IM-0001-0001.dcm`).

**Note:** This notebook focuses on single-slice inspection and analysis, not multi-slice DICOM series reconstruction.

## 1. Environment Setup and Project Structure

This section prepares the Python environment, imports the required libraries, and establishes the project directory and dataset paths.

In [ ]:
%pip install pydicom

In [ ]:
import pydicom
from pathlib import Path

print("pydicom version:", pydicom.__version__)
print("Libraries imported successfully.")

## 2. DICOM File Loading and Metadata Inspection

This section locates and loads the CT DICOM file using Pydicom. It examines the DICOM header and extracts relevant patient-independent imaging and acquisition metadata.

In [ ]:
project_path = Path.cwd().parent

print("Project directory:")
print(project_path)

In [ ]:
print("Project contents:")
for item in project_path.iterdir():
    print(item.name)

In [ ]:
notebook_path = project_path / "Notebooks"

print("Notebook folder contents:")
for item in notebook_path.iterdir():
    print(item.name)

In [ ]:
data_path = project_path / "Data"

print("Data folder contents:")
for item in data_path.iterdir():
    print(item.name)

In [ ]:
dicom_file = data_path / "IM-0001-0001.dcm"

dcm = pydicom.dcmread(dicom_file)

print("DICOM file loaded successfully.")
print(type(dcm))

## 3. DICOM Header and Metadata Exploration

This section examines the DICOM header, including its data elements, tags, value representations, and imaging-related metadata. It provides an overview of the information stored alongside the CT image.

In [ ]:
print(dcm)

## 4. Pixel Data Extraction and Image Characteristics

This section extracts the image pixel array from the DICOM dataset and examines its fundamental characteristics, including array dimensions, data type, and pixel intensity values.

In [ ]:
pixel_array = dcm.pixel_array

print("Pixel array extracted successfully.")
print("Array type:", type(pixel_array))
print("Array shape:", pixel_array.shape)
print("Array data type:", pixel_array.dtype)

In [ ]:
print("Minimum pixel value:", pixel_array.min())
print("Maximum pixel value:", pixel_array.max())

print("\nFirst 10 pixel values:")
print(pixel_array[0, :10])

print("\nPixel value at position (0, 0):", pixel_array[0, 0])
print("Pixel value at position (100, 200):", pixel_array[100, 200])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 8))

plt.imshow(pixel_array, cmap="gray")

plt.title("DICOM Image")
plt.axis("off")

plt.show()

In [ ]:
print("Image Rows:", dcm.Rows)
print("Image Columns:", dcm.Columns)

print("Pixel Spacing:", getattr(dcm, "PixelSpacing", "Not available"))
print("Slice Thickness:", getattr(dcm, "SliceThickness", "Not available"))

print("Image Position Patient:", getattr(dcm, "ImagePositionPatient", "Not available"))
print("Image Orientation Patient:", getattr(dcm, "ImageOrientationPatient", "Not available"))

In [ ]:
print("Rescale Slope:", getattr(dcm, "RescaleSlope", "Not available"))
print("Rescale Intercept:", getattr(dcm, "RescaleIntercept", "Not available"))
print("Rescale Type:", getattr(dcm, "RescaleType", "Not available"))

## 4. Pixel Data Conversion to Hounsfield Units (HU)

This section converts the raw CT pixel values into Hounsfield Units (HU) using the DICOM Rescale Slope and Rescale Intercept.

The converted values represent standardized CT attenuation measurements used for interpreting tissue and anatomical structures.

In [ ]:
# Convert raw pixel values to Hounsfield Units (HU)

rescale_slope = getattr(dcm, "RescaleSlope", 1)
rescale_intercept = getattr(dcm, "RescaleIntercept", 0)

ct_image = pixel_array * rescale_slope + rescale_intercept

print("CT image converted to Hounsfield Units (HU).")
print("Array shape:", ct_image.shape)
print("Data type:", ct_image.dtype)
print("Minimum HU:", ct_image.min())
print("Maximum HU:", ct_image.max())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.hist(ct_image.flatten(), bins=100, color="gray")

plt.title("CT Image Intensity Distribution")
plt.xlabel("Hounsfield Units (HU)")
plt.ylabel("Number of Pixels")

plt.grid(axis="y", alpha=0.3)
plt.show()

## 5. CT Intensity Distribution and Percentile Analysis

This section examines the distribution of Hounsfield Unit values across the CT image using statistical percentiles. It helps characterize the range and distribution of image intensities.

In [ ]:
import numpy as np

percentiles = [0, 1, 5, 25, 50, 75, 95, 99, 100]

print("CT Intensity Percentiles:")
for p in percentiles:
    value = np.percentile(ct_image, p)
    print(f"{p}%: {value:.2f} HU")

## 6. CT Image Windowing and Visualization

This section visualizes the CT image using different window settings to demonstrate how window width and window level affect the visibility of anatomical structures.

In [ ]:
# Visualize CT image using different window settings

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

windows = [
    ("Soft Tissue Window", 40, 400),
    ("Lung Window", -600, 1500),
    ("Bone Window", 300, 1500)
]

for ax, (title, window_level, window_width) in zip(axes, windows):

    vmin = window_level - window_width / 2
    vmax = window_level + window_width / 2

    ax.imshow(ct_image, cmap="gray", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 7. CT Image Summary and Characteristics

This section provides a consolidated summary of the CT image, including its dimensions, data type, intensity range, mean and median Hounsfield Unit values, and available DICOM window information.

In [ ]:
# Summarize the CT image characteristics

print("CT Image Summary")
print("-" * 30)

print(f"Image dimensions: {ct_image.shape}")
print(f"Data type: {ct_image.dtype}")
print(f"Minimum intensity: {ct_image.min():.2f} HU")
print(f"Maximum intensity: {ct_image.max():.2f} HU")
print(f"Mean intensity: {ct_image.mean():.2f} HU")
print(f"Median intensity: {np.median(ct_image):.2f} HU")

print("\nDICOM Window Information")
print("-" * 30)

print(f"Window Center: {getattr(dcm, 'WindowCenter', 'Not available')}")
print(f"Window Width: {getattr(dcm, 'WindowWidth', 'Not available')}")

## 8. CT Acquisition Parameters

This section examines the technical parameters used during CT image acquisition, including scanner information, tube voltage, tube current, exposure settings, radiation dose indicators, reconstruction kernel, and patient positioning.

In [ ]:
# CT Acquisition Parameters

print("CT ACQUISITION PARAMETERS")
print("-" * 35)

print("Modality:", getattr(dcm, "Modality", "Not available"))
print("Manufacturer:", getattr(dcm, "Manufacturer", "Not available"))
print("Scanner Model:", getattr(dcm, "ManufacturerModelName", "Not available"))

print("Tube Voltage (kVp):", getattr(dcm, "KVP", "Not available"))
print("Tube Current (mA):", getattr(dcm, "XRayTubeCurrent", "Not available"))
print("Exposure Time (ms):", getattr(dcm, "ExposureTime", "Not available"))
print("Exposure (mAs):", getattr(dcm, "Exposure", "Not available"))

print("CTDIvol:", getattr(dcm, "CTDIvol", "Not available"))
print("DLP:", getattr(dcm, "DLP", "Not available"))

print("Reconstruction Kernel:", getattr(dcm, "ConvolutionKernel", "Not available"))
print("Image Type:", getattr(dcm, "ImageType", "Not available"))
print("Patient Position:", getattr(dcm, "PatientPosition", "Not available"))

In [ ]:
# Check important DICOM metadata fields

important_tags = [
    "PatientID",
    "StudyInstanceUID",
    "SeriesInstanceUID",
    "SOPInstanceUID",
    "StudyDate",
    "SeriesDate",
    "Modality",
    "Manufacturer",
    "ManufacturerModelName",
    "Rows",
    "Columns",
    "PixelSpacing",
    "SliceThickness",
    "ImagePositionPatient",
    "ImageOrientationPatient",
    "WindowCenter",
    "WindowWidth",
    "RescaleSlope",
    "RescaleIntercept",
    "PatientPosition",
    "KVP",
    "Exposure",
    "CTDIvol",
    "DLP"
]

print("DICOM METADATA COMPLETENESS CHECK")
print("-" * 40)

for tag in important_tags:
    value = getattr(dcm, tag, None)

    if value is None:
        print(f"{tag}: MISSING")
    else:
        print(f"{tag}: PRESENT")

In [ ]:
# PIXEL ARRAY INSPECTION

print("PIXEL ARRAY INSPECTION")
print("-" * 35)

print("Array shape:", pixel_array.shape)
print("Number of dimensions:", pixel_array.ndim)
print("Data type:", pixel_array.dtype)
print("Total number of pixels:", pixel_array.size)

print("Contains NaN values:", np.isnan(pixel_array).any())
print("Contains infinite values:", np.isinf(pixel_array).any())

print("Unique values:", len(np.unique(pixel_array)))

In [ ]:
# RAW PIXEL VALUE INSPECTION

print("RAW PIXEL VALUE INSPECTION")
print("-" * 35)

print("Minimum raw pixel value:", pixel_array.min())
print("Maximum raw pixel value:", pixel_array.max())
print("Mean raw pixel value:", pixel_array.mean())
print("Median raw pixel value:", np.median(pixel_array))

print("\nSample raw pixel values:")
print(pixel_array[0:5, 0:5])

In [ ]:
# PIXEL VALUE SAMPLES FROM DIFFERENT REGIONS
# Inspect raw pixel values from different parts of the CT image

print("PIXEL VALUE SAMPLES FROM DIFFERENT REGIONS")
print("-" * 40)

# Sample pixels from the top-left corner of the image
# This region is outside the patient's body and contains air
print("Top-left corner:")
print(pixel_array[0:5, 0:5])

# Sample pixels from the central region of the image
# This region is likely to contain anatomical structures
print("\nCenter region:")
print(pixel_array[250:255, 250:255])

# Sample pixels from the upper-middle region
print("\nUpper-middle region:")
print(pixel_array[100:105, 250:255])

# Sample pixels from the lower-middle region
print("\nLower-middle region:")
print(pixel_array[400:405, 250:255])

In [ ]:
# REGIONAL PIXEL INTENSITY ANALYSIS
# Divide the CT image into regions and calculate the mean pixel intensity
# in each region to examine the spatial distribution of intensities.

height, width = pixel_array.shape

regions = {
    "Top-left": pixel_array[:height//2, :width//2],
    "Top-right": pixel_array[:height//2, width//2:],
    "Bottom-left": pixel_array[height//2:, :width//2],
    "Bottom-right": pixel_array[height//2:, width//2:],
    "Center": pixel_array[
        height//2-50:height//2+50,
        width//2-50:width//2+50
    ]
}

print("REGIONAL PIXEL INTENSITY ANALYSIS")
print("-" * 40)

for name, region in regions.items():
    print(f"{name}:")
    print(f"  Mean intensity: {region.mean():.2f}")
    print(f"  Minimum intensity: {region.min()}")
    print(f"  Maximum intensity: {region.max()}")
    print()

In [ ]:
# REGIONAL GRID VISUALIZATION
# Display the CT image with lines dividing it into four quadrants
# and marking the central region used in the previous analysis.

plt.figure(figsize=(8, 8))

plt.imshow(ct_image, cmap="gray")
plt.axhline(height // 2, color="red", linestyle="--", linewidth=1)
plt.axvline(width // 2, color="red", linestyle="--", linewidth=1)

# Mark the central 100 x 100 pixel region
plt.axhline(height // 2 - 50, color="yellow", linestyle=":", linewidth=1)
plt.axhline(height // 2 + 50, color="yellow", linestyle=":", linewidth=1)
plt.axvline(width // 2 - 50, color="yellow", linestyle=":", linewidth=1)
plt.axvline(width // 2 + 50, color="yellow", linestyle=":", linewidth=1)

plt.title("CT Image - Regional Analysis Grid")
plt.axis("off")
plt.show()

In [ ]:
# REGIONAL CT INTENSITY ANALYSIS IN HOUNSFIELD UNITS
# Repeat the regional analysis using the rescaled CT image (ct_image),
# rather than raw pixel values.

height, width = ct_image.shape

regions_hu = {
    "Top-left": ct_image[:height//2, :width//2],
    "Top-right": ct_image[:height//2, width//2:],
    "Bottom-left": ct_image[height//2:, :width//2],
    "Bottom-right": ct_image[height//2:, width//2:],
    "Center": ct_image[
        height//2-50:height//2+50,
        width//2-50:width//2+50
    ]
}

print("REGIONAL CT INTENSITY ANALYSIS (HU)")
print("-" * 45)

for name, region in regions_hu.items():
    print(f"{name}:")
    print(f"  Mean HU: {region.mean():.2f}")
    print(f"  Median HU: {np.median(region):.2f}")
    print(f"  Minimum HU: {region.min():.2f}")
    print(f"  Maximum HU: {region.max():.2f}")
    print()

## 9. Conclusion

This notebook demonstrated the inspection and analysis of a CT image stored in DICOM format.

The following tasks were completed:

- DICOM file loading and metadata inspection.
- Examination of imaging and acquisition parameters.
- Extraction and inspection of the pixel array.
- Conversion of raw pixel values into Hounsfield Units (HU).
- Statistical analysis of CT image intensities.
- CT windowing and image visualisation.
- Regional pixel intensity analysis.

### Limitations

The analysis was performed using a single CT DICOM image. Multi-slice reconstruction and volumetric analysis were outside the scope of this notebook.

The extracted metadata and image characteristics provide a foundation for subsequent DICOM preprocessing, image analysis and viewer development.